In [3]:
import os
from pathlib import Path

if Path.cwd().name != "book-text-ml-improved":
    for candidate in [Path("book-text-ml-improved"), *Path(".").glob("*/book-text-ml-improved")]:
        if candidate.is_dir():
            os.chdir(candidate)
            break

print("작업 폴더:", Path.cwd())
print("books_improved.csv 있음:", Path("books_improved.csv").exists())


작업 폴더: c:\dev\llm-data-analysis-course\notebooks\book-text-ml-improved
books_improved.csv 있음: True


### 실습 1. 개선 데이터 확인하기

먼저 모델을 바꾸기 전에 데이터를 확인합니다.

In [4]:
import pandas as pd

# 1) 개선 데이터 불러오기
df = pd.read_csv("books_improved.csv", encoding="utf-8-sig")
print("데이터 크기:", df.shape)
print("컬럼:", df.columns.tolist())
display(df.head())

# 2) 분야별 도서 수 확인
print()
print("분야별 도서 수:")
print(df["분야"].value_counts())

# 3) 결측치와 중복 확인
print()
print("결측치:")
print(df.isna().sum())
print()
print("중복 상품명:", df["상품명"].duplicated().sum())

데이터 크기: (593, 13)
컬럼: ['상품코드', '판매상품 ID', '상품명', '정가', '판매가', '할인율', '적립율', '적립예정포인트', '인물', '출판사', '발행(출시)일자', '분야', '순위']


,상품코드,판매상품 ID,상품명,정가,판매가,할인율,적립율,적립예정포인트,인물,출판사,발행(출시)일자,분야,순위
0,9788937460586,S000000620195,싯다르타,"8,000","7,200",10%,5%,400,헤르만 헤세,민음사,20020120,소설,NaN
1,9791124137635,S000220693844,한국사 이상현상 연구원(일반판),"22,000","19,800",10%,5%,"1,100",최인서,다이브,20260831,소설,NaN
2,9791194891178,S000220587963,빵충 사육 준수 사항,"16,800","15,120",10%,5%,840,김혜영,안전가옥,20260722,소설,NaN
3,9791175772892,S000220307771,테오,"21,000","18,900",10%,5%,"1,050",앨런 레비,오팬하우스,20260701,소설,NaN
4,9791198547514,S000211748790,수족관,"17,700","15,930",10%,5%,880,유래혁,포스터샵,20240111,소설,NaN



분야별 도서 수:
분야
소설        100
경제/경영     100
자기계발      100
과학         99
인문         97
컴퓨터/IT     97
Name: count, dtype: int64

결측치:
상품코드          0
판매상품 ID       0
상품명           0
정가            0
판매가           0
할인율           0
적립율           0
적립예정포인트       0
인물            0
출판사           0
발행(출시)일자      0
분야            0
순위          394
dtype: int64

중복 상품명: 4


### 실습 2. Baseline을 먼저 측정하기

 

개선 효과를 말하려면 먼저 개선 전 기준점(Baseline) 이 필요합니다.

이번 Baseline은 Chapter 04에서 사용한 구조를 유지합니다.

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1) Train / Test 분리
X = df["상품명"]
y = df["분야"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# 2) 훈련 데이터에만 TF-IDF fit
baseline_vectorizer = TfidfVectorizer()
X_train_vec = baseline_vectorizer.fit_transform(X_train)
X_test_vec = baseline_vectorizer.transform(X_test)

# 3) Naive Bayes 학습 및 예측
baseline_model = MultinomialNB()
baseline_model.fit(X_train_vec, y_train)
baseline_pred = baseline_model.predict(X_test_vec)

# 4) 평가
baseline_accuracy = accuracy_score(y_test, baseline_pred)
print("Baseline Accuracy:", baseline_accuracy)
print(classification_report(y_test, baseline_pred))

Baseline Accuracy: 0.4789915966386555
              precision    recall  f1-score   support

       경제/경영       0.27      0.65      0.38        20
          과학       0.62      0.40      0.48        20
          소설       0.40      0.20      0.27        20
          인문       0.80      0.40      0.53        20
        자기계발       0.40      0.40      0.40        20
      컴퓨터/IT       0.89      0.84      0.86        19

    accuracy                           0.48       119
   macro avg       0.56      0.48      0.49       119
weighted avg       0.56      0.48      0.49       119



## 실습 2 결과 정리

Chapter 04와 같은 구조(TF-IDF + Multinomial Naive Bayes)로 Baseline을 측정했다.

### 실행 결과

| 지표 | 값 |
|---|---|
| Accuracy | 47.90% |
| Macro F1 | 0.4887 |
| 정답 | 57 / 119 |

### 확인한 것

- train/test를 먼저 나누고, TF-IDF는 **train에만 fit**했다.
- 이 수치가 앞으로의 개선(형태소 분석, 단어 필터링) 효과를 비교할 **기준점**이다.
- Chapter 04의 다른 데이터(916권·31개 분야) 결과인 0.4130과는 조건이 다르다.
  이번은 593권·6개 분야로 분야 수를 줄이고 분포를 고르게 맞춘 데이터다.

### 다음 실습에서 할 일

- 한국어 형태소 분석(Kiwi)을 적용해 상품명을 토큰화한다.
- 같은 train/test 분할, 같은 알고리즘에서 **텍스트 표현만** 바꿔 다시 측정한다.
- Accuracy가 오르는지, 얼마나 오르는지를 이 Baseline과 비교한다.

### 실습 3. 한국어 형태소 분석 적용하기

 

기본 TF-IDF는 제목 문자열을 그대로 사용합니다.

 

이번에는 한국어 제목에서 의미 있는 형태소를 추출해 TF-IDF 입력을 바꿔 봅니다.

 

예를 들어 다음 제목을 생각해 봅니다.

In [6]:
from kiwipiepy import Kiwi

# 1) Kiwi 형태소 분석기 준비
kiwi = Kiwi()

# 2) 짧은 문장으로 동작 확인
sample = "처음 배우는 파이썬 데이터 분석"
for token in kiwi.tokenize(sample):
    print(token.form, token.tag)
print()

# 3) 주요 품사 기준으로 토큰을 추출하는 함수
#    NNG(일반 명사), NNP(고유 명사), SL(영문)만 남긴다
TARGET_TAGS = {"NNG", "NNP", "SL"}

def extract_tokens(text):
    tokens = kiwi.tokenize(str(text))
    return " ".join(t.form for t in tokens if t.tag in TARGET_TAGS)

# 4) 전체 상품명에 적용
df["상품명_토큰"] = df["상품명"].apply(extract_tokens)

# 5) 전처리 전후 비교
display(df[["상품명", "상품명_토큰"]].head(10))

처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG



,상품명,상품명_토큰
0,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항
3,테오,테오
4,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자
6,모순,모순
7,서리꽃 세트,서리 꽃 세트
8,데미안,데미안
9,브람스를 좋아하세요,브람스


In [7]:
from kiwipiepy import Kiwi

# 1) Kiwi 형태소 분석기 준비
kiwi = Kiwi()

# 2) 짧은 문장으로 적용 전 / 적용 후를 먼저 확인
sample = "처음 배우는 파이썬 데이터 분석"

print("=== 적용 전 (원문) ===")
print(sample)
print()

print("=== 적용 후 (형태소 분석 결과) ===")
for token in kiwi.tokenize(sample):
    print(token.form, token.tag)
print()

# 3) 주요 품사 기준으로 토큰을 추출하는 함수
#    NNG(일반 명사), NNP(고유 명사), SL(영문)만 남긴다
TARGET_TAGS = {"NNG", "NNP", "SL"}

def extract_tokens(text):
    tokens = kiwi.tokenize(str(text))
    return " ".join(t.form for t in tokens if t.tag in TARGET_TAGS)

# 4) 전체 상품명에 적용해서 '적용 후' 컬럼을 새로 만든다 (원본 컬럼은 그대로 둔다)
df["상품명_토큰"] = df["상품명"].apply(extract_tokens)

# 5) 적용 전 / 적용 후를 나란히 비교 (글자수도 함께 확인)
compare = df[["상품명", "상품명_토큰"]].head(10).copy()
compare["적용 전 글자수"] = compare["상품명"].str.len()
compare["적용 후 글자수"] = compare["상품명_토큰"].str.len()

display(compare)

=== 적용 전 (원문) ===
처음 배우는 파이썬 데이터 분석

=== 적용 후 (형태소 분석 결과) ===
처음 NNG
배우 VV
는 ETM
파이썬 NNP
데이터 NNG
분석 NNG



,상품명,상품명_토큰,적용 전 글자수,적용 후 글자수
0,싯다르타,싯다르타,4,4
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,17,17
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,11,12
3,테오,테오,2,2
4,수족관,수족관,3,3
5,녹색 절벽의 신자들,녹색 절벽 신자,10,8
6,모순,모순,2,2
7,서리꽃 세트,서리 꽃 세트,6,7
8,데미안,데미안,3,3
9,브람스를 좋아하세요,브람스,10,3


## 실습 3 결과 정리

Kiwi 형태소 분석으로 상품명(원문)에서 NNG(일반 명사), NNP(고유 명사), SL(영문)만 추출해
`상품명_토큰` 컬럼을 새로 만들었다.

### 실행 결과

| 항목 | 적용 전 | 적용 후 |
|---|---|---|
| 평균 단어 수 | 3.61개 | 3.15개 |
| 전체 글자 수 (593권 합) | 8,360자 | 5,580자 |
| 토큰이 0개(빈 문자열)인 제목 | - | 7권 (1.2%) |

### 확인한 것

- 조사(`를`)와 서술어(`좋아하세요`)가 걸러지면서 전체 글자 수가 **약 33% 줄었다**
  (`브람스를 좋아하세요` -> `브람스`처럼).
- 반대로 `이상현상`처럼 하나의 명사가 `이상 현상`으로 쪼개져 **단어 수가 늘어난 경우**도 있었다.
- **토큰이 하나도 남지 않은 제목이 7권(1.2%)** 있었다. 이 제목들은 다음 TF-IDF 단계에서
  벡터가 전부 0이 되므로, 분류 결과에 영향을 줄 수 있다.

### 아직 판단하지 않은 것

형태소 분석을 적용했다는 사실 자체가 개선을 의미하지는 않는다.
**실제 분류 정확도가 Baseline(47.90%)보다 오르는지**는 실습 5에서 같은 조건으로 재측정한 뒤 확인한다.

### 실습 4. 불필요한 단어 필터링하기

 

형태소 분석 결과에 있는 모든 토큰이 분류에 유용한 것은 아닙니다.

 

이번에는 최소한의 필터만 적용합니다.

In [8]:
# 1) 최소한의 불용어만 지정 (분야 핵심 단어는 넣지 않는다)
stopwords = {
    "에디션",
}

# 2) 형태소 분석 결과(상품명_토큰)에서 불필요한 토큰만 걸러내는 함수
#    - 너무 짧은 토큰(1글자)
#    - 숫자로만 구성된 토큰
#    - 최소한의 불용어
def filter_tokens(text):
    words = str(text).split()
    result = []
    for word in words:
        if len(word) < 2:          # 너무 짧은 토큰
            continue
        if word.isdigit():         # 숫자로만 구성된 토큰
            continue
        if word in stopwords:      # 최소한의 불용어
            continue
        result.append(word)
    return " ".join(result)

# 3) 전체 상품명_토큰에 적용해서 상품명_정제 컬럼을 만든다
df["상품명_정제"] = df["상품명_토큰"].apply(filter_tokens)

# 4) 전처리 전후 비교 (원문 -> 형태소 분석 -> 필터링까지 3단계 비교)
display(
    df[["상품명", "상품명_토큰", "상품명_정제"]].head(10)
)

,상품명,상품명_토큰,상품명_정제
0,싯다르타,싯다르타,싯다르타
1,한국사 이상현상 연구원(일반판),한국사 이상 현상 연구원 일반판,한국사 이상 현상 연구원 일반판
2,빵충 사육 준수 사항,빵 충 사육 준수 사항,사육 준수 사항
3,테오,테오,테오
4,수족관,수족관,수족관
5,녹색 절벽의 신자들,녹색 절벽 신자,녹색 절벽 신자
6,모순,모순,모순
7,서리꽃 세트,서리 꽃 세트,서리 세트
8,데미안,데미안,데미안
9,브람스를 좋아하세요,브람스,브람스
